# Training Pipeline
[run_training_dpo_pipeline.ipynb](https://github.com/shibing624/MedicalGPT/blob/main/notebooks/run_training_dpo_pipeline.ipynb)    | [Open In Colab](https://colab.research.google.com/github/zhifeiluo7/MedicalGPT_Project/blob/main/notebooks/run_training_dpo_pipeline.ipynb)

# 虚拟环境预处理

配置一个干净的虚拟环境，避免Colab的煞笔环境冲突

In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [2]:
!conda create -n medgpt python=3.10 -y
!source activate medgpt

Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.5.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/medgpt

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.5.20  |       hbd8a1cb_0         127 KB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.8.1             |       hecca717_0          75 KB  conda-forge
    libffi-3.5.2               |       h3435931_0          57 KB  conda-forge
  

In [3]:
!conda install pip -y
!pip install transformers peft torch torchvision torchaudio

Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / - \ done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - pip


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    certifi-2026.5.20          |     pyhd8ed1ab_0         131 KB  conda-forge
    conda-24.11.3              |  py311h38be061_0         1.1 MB  conda-forge
    ------------------------------------------------------------
                                           Total:         1.3 MB

The following packages will be UPDATED:

  ca-certificates    conda-forge/linux-64::ca-certificates~ --> conda-forge/noarch::ca-certificates-2026.5.20-hbd8a1cb_0 
  certifi                           2024.12.14-pyhd8ed1ab_0 --> 2026.5.20-pyhd8ed1ab_0 
  conda                             24.11.2-py311h38be061_1 --> 24.11.3-py311h38be061_0 
  openssl                                  3.4.0-

# Stage 1: Continue Pretraining

第一阶段：PT(Continue PreTraining)增量预训练，在海量领域文本数据上二次预训练GPT模型，以适配领域数据分布

注意：
1. 此阶段是可选的，如果你没有海量领域文本，可以跳过此阶段，直接进行SFT阶段的有监督微调
2. 我实验发现：做领域知识注入，SFT比PT更高效，也可以跳过PT阶段

| Stage 1: Continue Pretraining   |  [pretraining.py](https://github.com/shibing624/MedicalGPT/blob/main/training/pretraining.py) | [run_pt.sh](https://github.com/shibing624/MedicalGPT/blob/main/scripts/run_pt.sh)    |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是Qwen/Qwen2.5-0.5B
2. 数据集：PT阶段使用的是中文天龙八部小说部分文本和英文书籍部分文本，位于`data/pretrain`文件夹

## 配置运行环境

本地执行可注释以下配置环境的命令，colab执行要打开注释，用于配置环境

colab建议使用T4 GPU训练，设置方式：`代码执行程序 -> 更改运行时类型 -> 运行时类型：Python3，硬件加速器：GPU，GPU类型：T4 -> 保存`

步骤：
1. 下载最新代码到本地
2. 安装依赖包

依赖包如下，保证最新版本：

```
loguru
transformers
sentencepiece
datasets
tensorboard
tqdm
peft
trl
torchao> 0.16.0
```

In [ ]:
!git clone --depth 1 https://github.com/zhifeiluo7/MedicalGPT_Project.git
%cd MedicalGPT
%ls
!pip install --upgrade "torchao>0.16.0"
!pip install -r requirements.txt

fatal: destination path 'MedicalGPT_Project' already exists and is not an empty directory.
/content/MedicalGPT
CITATION.cff     demo/       MedicalGPT/     README.md         tests/
_config.yml      DISCLAIMER  notebooks/      requirements.txt  tools/
CONTRIBUTING.md  docs/       outputs-pt-v1/  role_play_data/   training/
data/            LICENSE     README_EN.md    scripts/
  Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (20 kB)
Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (3.2 MB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
  Using cached trl-1.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached latex2sympy2_extended-1.11.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached math_verify-0.5.2-py3-none-any.whl.metadata (347 bytes)
  Using cached latex2sympy2_extended-1.0.6-py3-none-any.whl.metadata (4.9 kB)
  Using

### Stage1 PT

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

**以下参数可以根据GPU实际情况修改，当前参数是根据Colab的T4单卡GPU（16GB显存）配置的**

In [ ]:
%ls ./data/pretrain/

en_article_tail500.txt  fever.txt  tianlongbabu.txt


In [ ]:
!python training/pretraining.py \
    --model_name_or_path Qwen/Qwen2.5-0.5B \
    --train_file_dir ./data/pretrain \
    --validation_file_dir ./data/pretrain \
    --per_device_train_batch_size 3 \
    --per_device_eval_batch_size 3 \
    --do_train \
    --do_eval \
    --use_peft True \
    --seed 42 \
    --bf16 \
    --max_train_samples 20000 \
    --max_eval_samples 10 \
    --num_train_epochs 1 \
    --learning_rate 2e-4 \
    --warmup_ratio 0.05 \
    --weight_decay 0.01 \
    --logging_strategy steps \
    --logging_steps 10 \
    --eval_steps 50 \
    --eval_strategy steps \
    --save_steps 50 \
    --save_strategy steps \
    --save_total_limit 3 \
    --gradient_accumulation_steps 1 \
    --preprocessing_num_workers 1 \
    --block_size 128 \
    --output_dir outputs-pt-v1 \
    --overwrite_output_dir \
    --ddp_timeout 30000 \
    --logging_first_step True \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --device_map auto \
    --report_to tensorboard \
    --ddp_find_unused_parameters False \
    --gradient_checkpointing True

W0607 11:52:26.157000 15472 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0607 11:52:26.185000 15472 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
2026-06-07 11:52:26.650 | INFO     | __main__:main:364 - Model args: ModelArguments(model_name_or_path='Qwen/Qwen2.5-0.5B', tokenizer_name_or_path=None, load_in_8bit=False, load_in_4bit=False, cache_dir=None, model_revision='main', hf_hub_token=None, use_fast_tokenizer=False, torch_dtype='bfloat16', device_map='au

In [ ]:
%ls -lh outputs-pt-v1

total 28M
-rw-r--r-- 1 root root 1.1K Jun  7 12:03 adapter_config.json
-rw-r--r-- 1 root root  17M Jun  7 12:03 adapter_model.safetensors
-rw-r--r-- 1 root root  470 Jun  7 12:03 all_results.json
-rw-r--r-- 1 root root 2.4K Jun  7 12:03 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Jun  7 12:02 checkpoint-750/
drwxr-xr-x 2 root root 4.0K Jun  7 12:02 checkpoint-800/
drwxr-xr-x 2 root root 4.0K Jun  7 12:03 checkpoint-845/
-rw-r--r-- 1 root root  262 Jun  7 12:03 eval_results.json
-rw-r--r-- 1 root root 5.1K Jun  7 12:03 README.md
drwxr-xr-x 4 root root 4.0K Jun  7 11:52 runs/
-rw-r--r-- 1 root root  723 Jun  7 12:03 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun  7 12:03 tokenizer.json
-rw-r--r-- 1 root root  21K Jun  7 12:03 trainer_state.json
-rw-r--r-- 1 root root  228 Jun  7 12:03 train_results.json


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`tools/merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [ ]:
!sed -i '/AutoModelForConditionalGeneration/d' /content/MedicalGPT/tools/merge_peft_adapter.py

In [ ]:
!python tools/merge_peft_adapter.py \
    --base_model Qwen/Qwen2.5-0.5B --lora_model outputs-pt-v1 --output_dir merged-pt/

W0607 12:13:43.455000 20980 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0607 12:13:43.582000 20980 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Namespace(base_model='Qwen/Qwen2.5-0.5B', tokenizer_path=None, lora_model='outputs-pt-v1', resize_emb=False, output_dir='merged-pt/', hf_hub_model_id='', hf_hub_token=None)
Base model: Qwen/Qwen2.5-0.5B
LoRA model: outputs-pt-v1
Loading LoRA for causal language model (archs=['Qwen2ForCausalLM'])
Loading weights: 100% 290/290 [00:00<00:00, 448.51it/s]
Merging with merge_and_unload...
Saving to Hugging Face forma

In [ ]:
%ls -lh merged-pt/

total 954M
-rw-r--r-- 1 root root 2.4K Jun  7 12:13 chat_template.jinja
-rw-r--r-- 1 root root 1.3K Jun  7 12:13 config.json
-rw-r--r-- 1 root root  139 Jun  7 12:13 generation_config.json
-rw-r--r-- 1 root root 943M Jun  7 12:13 model.safetensors
-rw-r--r-- 1 root root  697 Jun  7 12:13 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun  7 12:13 tokenizer.json


In [ ]:
%cat merged-pt/config.json

{
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "pad_token_id": n

Stage1 增量预训练完成。

# Stage 2: Supervised FineTuning

第二阶段：SFT(Supervised Fine-tuning)有监督微调，构造指令微调数据集，在预训练模型基础上做指令精调，以对齐指令意图，并注入领域知识

| Stage 2: Supervised Fine-tuning | [supervised_finetuning.py](https://github.com/shibing624/MedicalGPT/blob/main/training/supervised_finetuning.py) | [run_sft.sh](https://github.com/shibing624/MedicalGPT/blob/main/scripts/run_sft.sh)  |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是Qwen/Qwen2.5-0.5B 或者 Stage1得到的预训练模型
2. 数据集：SFT阶段使用的是使用的是Belle的1千条抽样数据，位于`data/finetune`文件夹

## Stage2 咱们开始吧

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

In [27]:
%ls ./data/sft

glaive_toolcall_zh_demo.jsonl  sharegpt_zh_1K_format.jsonl
medical_sft_1K_format.jsonl


In [28]:
!python training/supervised_finetuning.py \
    --model_name_or_path merged-pt \
    --train_file_dir ./data/sft \
    --validation_file_dir ./data/sft \
    --per_device_train_batch_size 4 \
    --per_device_eval_batch_size 4 \
    --do_train \
    --do_eval \
    --use_peft True \
    --bf16 \
    --max_train_samples 1000 \
    --max_eval_samples 10 \
    --num_train_epochs 1 \
    --learning_rate 2e-5 \
    --warmup_ratio 0.05 \
    --weight_decay 0.05 \
    --logging_strategy steps \
    --logging_steps 10 \
    --eval_steps 50 \
    --eval_strategy steps \
    --save_steps 500 \
    --save_strategy steps \
    --save_total_limit 3 \
    --gradient_accumulation_steps 1 \
    --preprocessing_num_workers 1 \
    --output_dir outputs-sft-v1 \
    --ddp_timeout 30000 \
    --logging_first_step True \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --device_map auto \
    --report_to tensorboard \
    --ddp_find_unused_parameters False \
    --gradient_checkpointing True

W0607 13:52:30.659000 46792 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0607 13:52:30.684000 46792 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-06-07 13:52:30.963 | WARNING  | __main__:__post_init__:191 - You may set max_train_samples = -1 to run all samples in production.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
2026-06-07 13:52:31.164 | INFO     | __main__:main:352 - Model args: ModelArguments(model_name_or_path='merged-pt', load_in_8bit=False, load_in_4bit=False, tokenizer_name_or_path=N

In [29]:
%ls -lh outputs-sft-v1

total 28M
-rw-r--r-- 1 root root 1.1K Jun  7 14:05 adapter_config.json
-rw-r--r-- 1 root root  17M Jun  7 14:05 adapter_model.safetensors
-rw-r--r-- 1 root root  428 Jun  7 14:05 all_results.json
-rw-r--r-- 1 root root 2.4K Jun  7 14:05 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Jun  7 14:05 checkpoint-250/
-rw-r--r-- 1 root root  220 Jun  7 14:05 eval_results.json
-rw-r--r-- 1 root root 5.1K Jun  7 14:05 README.md
drwxr-xr-x 3 root root 4.0K Jun  7 13:52 runs/
-rw-r--r-- 1 root root  733 Jun  7 14:05 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun  7 14:05 tokenizer.json
-rw-r--r-- 1 root root 6.3K Jun  7 14:05 trainer_state.json
-rw-r--r-- 1 root root  228 Jun  7 14:05 train_results.json


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`tools/merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [30]:
!python tools/merge_peft_adapter.py \
    --base_model merged-pt --lora_model outputs-sft-v1 --output_dir ./merged-sft

W0607 14:29:21.318000 56263 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0607 14:29:21.355000 56263 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Namespace(base_model='merged-pt', tokenizer_path=None, lora_model='outputs-sft-v1', resize_emb=False, output_dir='./merged-sft', hf_hub_model_id='', hf_hub_token=None)
Base model: merged-pt
LoRA model: outputs-sft-v1
Loading LoRA for causal language model (archs=['Qwen2ForCausalLM'])
Loading weights: 100% 290/290 [00:00<00:00, 302.04it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing

In [31]:
%ls -lh merged-sft/

total 954M
-rw-r--r-- 1 root root 2.4K Jun  7 12:13 chat_template.jinja
-rw-r--r-- 1 root root 1.3K Jun  7 14:29 config.json
-rw-r--r-- 1 root root  139 Jun  7 12:13 generation_config.json
-rw-r--r-- 1 root root 943M Jun  7 14:29 model.safetensors
-rw-r--r-- 1 root root  697 Jun  7 12:13 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun  7 12:13 tokenizer.json


In [32]:
%cat merged-sft/config.json

{
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "pad_token_id": n

Stage2 SFT训练完成。

# Stage 3: DPO(Direct Preference Optimization)

第三阶段：DPO(Direct Preference Optimization)直接偏好优化，DPO通过直接优化语言模型来实现对其行为的精确控制，而无需使用复杂的强化学习，也可以有效学习到人类偏好，DPO相较于RLHF更容易实现且易于训练，效果更好

| Stage 3: Direct Preference Optimization        |  [dpo_training.py](https://github.com/shibing624/MedicalGPT/blob/main/training/dpo_training.py) | [run_dpo.sh](https://github.com/shibing624/MedicalGPT/blob/main/scripts/run_dpo.sh)    |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是`Qwen/Qwen2.5-0.5B` 或者 Stage2得到的SFT模型
2. 数据集：DPO阶段使用的是医疗reward数据，抽样了500条，位于`data/reward`文件夹

## Stage3 咱们开始吧

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

In [33]:
%ls ./data/reward/

dpo_zh_500.jsonl  toolcall_dpo_zh_demo.jsonl


In [1]:
!python training/dpo_training.py \
    --model_name_or_path ./merged-sft \
    --template_name qwen \
    --train_file_dir ./data/reward \
    --validation_file_dir ./data/reward \
    --per_device_train_batch_size 3 \
    --per_device_eval_batch_size 1 \
    --do_train \
    --do_eval \
    --use_peft True \
    --max_train_samples 1000 \
    --max_eval_samples 500 \
    --max_steps 100 \
    --eval_steps 10 \
    --save_steps 50 \
    --max_source_length 256 \
    --max_target_length 256 \
    --output_dir outputs-dpo-v1 \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --bf16 True \
    --fp16 False \
    --device_map auto \
    --report_to tensorboard \
    --remove_unused_columns False \
    --gradient_checkpointing True \
    --cache_dir ./cache

python3: can't open file '/content/training/dpo_training.py': [Errno 2] No such file or directory


In [35]:
%ls -lh outputs-dpo-v1

total 4.0K
drwxr-xr-x 3 root root 4.0K Jun  7 14:30 runs/


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`tools/merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [ ]:
!python tools/merge_peft_adapter.py \
    --base_model merged-sft --lora_model outputs-dpo-v1 --output_dir merged-dpo/

In [ ]:
%ls -lh merged-dpo/

In [ ]:
%cat merged-dpo/config.json

Stage3 偏好建模第一次训练完成。

**至此一个完整的训练流程演示完成。**

# Test

In [ ]:
!python demo/inference.py --base_model merged-dpo
# 或在shell中运行
# python demo/inference.py --base_model merged-dpo --interactive

Input:介绍下南京
Response:  南京市位于江苏省西南部，是全国首批历史文化名城、国家中心城市和自由贸易试验区。

完。
